In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import lyricsgenius

In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR = BASE_DIR / "data" / "raw"

tracks = pd.read_csv(PROCESSED_DIR / "spotify_tracks_clean.csv")

tracks.head()


,track_id,track_name,album_id,album_name,track_number,duration_ms,explicit,spotify_url
0,6nt3AoYjkaqXMZhypTBky1,SWIM,6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,1,159007,False,https://open.spotify.com/track/6nt3AoYjkaqXMZh...
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,2,163226,False,https://open.spotify.com/track/7EytKcb3klVPpN5...
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,3,193813,False,https://open.spotify.com/track/5dZLsPskKzph16L...
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,4,219573,False,https://open.spotify.com/track/5AL5OrvyIMPqKjl...
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,5,153133,False,https://open.spotify.com/track/3MCJY7lXCHa0UNI...


In [3]:
load_dotenv(BASE_DIR / ".env")

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")

if not GENIUS_ACCESS_TOKEN:
    raise ValueError("GENIUS_ACCESS_TOKEN is missing from .env")

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True
)

genius.verbose = False

In [4]:
song = genius.search_song("Idol", "BTS")

print(song.title)
print(song.artist)
print(song.url)
print(song.lyrics[:1000])

IDOL
BTS
https://genius.com/Bts-idol-lyrics
You can call me artist (Artist)
You can call me idol (Idol)
아님 어떤 다른 (다른)
뭐라 해도, I don't care (I don't care)
I'm proud of it (Proud of it)
난 자유롭네
No more irony (Irony)
나는 항상 나였기에

손가락질 해 (Oh, yeah, yeah, yeah)
나는 전혀 신경 쓰지 않네
나를 욕하는 너의 (Woah)
그 이유가 뭐든 간에
I know what I am (I know what I am)
I know what I want (I know what I want)
I never gon' change (I never gon' change)
I never gon' trade (Trade off, choo-choo)

뭘 어쩌고 저쩌고 떠들어대셔?
(Talkin', talkin', talkin', bruh)
I do what I do, 그니까 넌 너나 잘하셔
(Dirty, dirty) You can't stop me lovin' myself

(Brrah, hoo, hoo) 얼쑤 좋다
You can't stop me lovin' myself
(Brrah, hoo, hoo) 지화자 좋다
You can't stop me lovin' myself

Oh, oh, ooh-woah (Hey)
Oh, oh, ooh-woah, woah
Oh, oh, ooh-woah (Brrah)
덩기덕 쿵더러러
 (
얼쑤
)
Oh, oh, ooh-woah (Hey)
Oh, oh, ooh-woah, woah
Oh, oh, ooh-woah (Brrah)
덩기덕 쿵더러러
 (
얼쑤
)

FACE OFF, 마치 오우상, ayy
Top star with that spotlight, ayy
때론 슈퍼히어로가 돼
돌려대 너의 Anpanman
Woah, 이십사시간이 적지
헷갈림, 내겐 사치 (Woah)
I do

In [5]:
test_songs = ["ON", "Dynamite", "Black Swan", "Blood Sweat & Tears"]

for title in test_songs:
    song = genius.search_song(title, "BTS")

    print("\n---")
    print("Search:", title)

    if song:
        print("Found:", song.title)
        print("Artist:", song.artist)
        print("URL:", song.url)
        print(song.lyrics[:200])
    else: 
        print("Not found")


---
Search: ON
Found: ON
Artist: BTS
URL: https://genius.com/Bts-on-lyrics
I can't understand what people are sayin'
어느 장단에 맞춰야 될지
한 발자국 떼면 한 발자국 커지는 shadow
잠에서 눈을 뜬 여긴 또 어디
어쩜 서울 또 New York or Paris
일어나니 휘청이는 몸

(Yeah) Look at my feet, look down
날 닮은 그림자
흔들리는 건 이놈인가
아니면 내 작

---
Search: Dynamite
Found: Dynamite
Artist: BTS
URL: https://genius.com/Bts-dynamite-lyrics
'Cause I, I, I'm in the stars tonight
So watch me bring the fire and set the night alight

Shoes on, get up in the morn'
Cup of milk, let's rock and roll
King Kong, kick the drum
Rolling on like a Rol

---
Search: Black Swan
Found: Black Swan
Artist: BTS
URL: https://genius.com/Bts-black-swan-lyrics
Do your thang
Do your thang with me now
Do your thang
Do your thang with me now
What's my thang?
What's my thang? Tell me now
Tell me now
Yeah, yeah, yeah, yeah

Ayy, 심장이 뛰지 않는대
더는 음악을 들을 때
Tryna pull

---
Search: Blood Sweat & Tears
Found: Blood, Sweat & Tears
Artist: Dreamville
URL: https://genius.com/Dreamville-bas-and-blac

In [6]:
from difflib import SequenceMatcher
import re
import unicodedata

def normalize_title(title):
    if pd.isna(title):
        return ""

    title = unicodedata.normalize("NFKC", str(title)).lower()
    title = title.replace("&", " and ")
    title = re.sub(r"[‘’´`]", "'", title)
    title = title.replace("[", "(").replace("]", ")")
    title = title.replace("{", "(").replace("}", ")")
    title = re.sub(r"[–—−]", "-", title)
    title = re.sub(r"[^0-9a-z가-힣ぁ-ゔァ-ヴー一-龯()\s/&-]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

def title_variants(title):
    normalized = normalize_title(title)

    if not normalized:
        return set()

    variants = {normalized}

    without_parentheses = re.sub(r"\([^)]*\)", " ", normalized)
    without_parentheses = re.sub(r"\s+", " ", without_parentheses).strip(" -")
    if without_parentheses:
        variants.add(without_parentheses)

    for part in re.findall(r"\(([^)]*)\)", normalized):
        part = re.sub(r"\s+", " ", part).strip(" -")
        if part:
            variants.add(part)
            for slash_part in part.split("/"):
                slash_part = slash_part.strip()
                if slash_part:
                    variants.add(slash_part)

    expanded = set()
    for variant in variants:
        simplified = variant
        simplified = re.sub(r"\bjapanese version\b", "japanese ver", simplified)
        simplified = re.sub(r"\bjapanese ver\.\b", "japanese ver", simplified)
        simplified = re.sub(r"\bversion\b", "ver", simplified)
        simplified = re.sub(r"\bver\.\b", "ver", simplified)
        simplified = re.sub(r"\bfull length edition\b", "full length", simplified)
        simplified = re.sub(r"\s+", " ", simplified).strip(" -")
        if simplified:
            expanded.add(simplified)
        for hyphen_part in re.split(r"\s+-\s+", simplified):
            hyphen_part = hyphen_part.strip()
            if hyphen_part:
                expanded.add(hyphen_part)

    return expanded

def similarity(a, b):
    variants_a = title_variants(a)
    variants_b = title_variants(b)

    if not variants_a or not variants_b:
        return 0

    return max(
        SequenceMatcher(None, left, right).ratio()
        for left in variants_a
        for right in variants_b
    )

RAW_VALID_ARTISTS = {
    "BTS",
    "RM",
    "RM (알엠)",
    "Jin",
    "Jin (진)",
    "SUGA",
    "SUGA (슈가)",
    "Agust D",
    "j-hope",
    "j-hope (제이홉)",
    "jhope",
    "Jimin",
    "Jimin (지민)",
    "V",
    "V (뷔)",
    "Taehyung",
    "Jung Kook",
    "Jung Kook (정국)",
    "Jungkook",
    "Jungkook (정국)"
}

RAW_TRANSLATION_ARTISTS = {
    "Genius Romanizations",
    "Genius English Translations"
}

def normalize_artist_name(artist_name):
    if pd.isna(artist_name):
        return ""

    artist_name = unicodedata.normalize("NFKC", str(artist_name)).lower()
    artist_name = artist_name.replace("&", " and ")
    artist_name = re.sub(r"[^0-9a-z가-힣ぁ-ゔァ-ヴー一-龯\s]", " ", artist_name)
    artist_name = re.sub(r"\s+", " ", artist_name).strip()
    return artist_name

VALID_ARTISTS = {normalize_artist_name(name) for name in RAW_VALID_ARTISTS}
TRANSLATION_ARTISTS = {normalize_artist_name(name) for name in RAW_TRANSLATION_ARTISTS}

def classify_artist(artist_name):
    normalized_artist = normalize_artist_name(artist_name)

    if normalized_artist in VALID_ARTISTS:
        return "valid"

    if normalized_artist in TRANSLATION_ARTISTS:
        return "translation"

    return "invalid"

In [7]:
test_songs = ["ON", "Dynamite", "Black Swan", "Blood Sweat & Tears"]

for title in test_songs:
    song = genius.search_song(title, "BTS")

    print("\n---")
    print("Search:", title)

    if song:
        title_score = similarity(title, song.title)
        artist_class = classify_artist(song.artist)
        artist_ok = artist_class == "valid"

        print("Found:", song.title)
        print("Artist:", song.artist)
        print("Title score:", round(title_score, 2))
        print("Artist OK:", artist_ok)
        print("URL:", song.url)

        if artist_class == "valid" and title_score >= 0.75:
            print("Status: exact")
        elif artist_class == "translation":
            print("Status: manual_review")
        elif artist_class == "valid" and title_score >= 0.45:
            print("Status: manual_review")
        else:
            print("Status: rejected")
    else:
        print("Status: not_found")


---
Search: ON
Found: ON
Artist: BTS
Title score: 1.0
Artist OK: True
URL: https://genius.com/Bts-on-lyrics
Status: exact

---
Search: Dynamite
Found: Dynamite
Artist: BTS
Title score: 1.0
Artist OK: True
URL: https://genius.com/Bts-dynamite-lyrics
Status: exact

---
Search: Black Swan
Found: Black Swan
Artist: BTS
Title score: 1.0
Artist OK: True
URL: https://genius.com/Bts-black-swan-lyrics
Status: exact

---
Search: Blood Sweat & Tears
Found: Blood, Sweat & Tears
Artist: Dreamville
Title score: 1.0
Artist OK: False
URL: https://genius.com/Dreamville-bas-and-black-sherif-blood-sweat-and-tears-lyrics
Status: rejected


In [8]:
RAW_VALID_ARTISTS = {
    "BTS",
    "RM",
    "RM (알엠)",
    "Jin",
    "Jin (진)",
    "SUGA",
    "SUGA (슈가)",
    "Agust D",
    "j-hope",
    "j-hope (제이홉)",
    "jhope",
    "Jimin",
    "Jimin (지민)",
    "V",
    "V (뷔)",
    "Taehyung",
    "Jung Kook",
    "Jung Kook (정국)",
    "Jungkook",
    "Jungkook (정국)"
}

RAW_TRANSLATION_ARTISTS = {
    "Genius Romanizations",
    "Genius English Translations"
}

def normalize_artist_name(artist_name):
    if pd.isna(artist_name):
        return ""

    artist_name = unicodedata.normalize("NFKC", str(artist_name)).lower()
    artist_name = artist_name.replace("&", " and ")
    artist_name = re.sub(r"[^0-9a-z가-힣ぁ-ゔァ-ヴー一-龯\s]", " ", artist_name)
    artist_name = re.sub(r"\s+", " ", artist_name).strip()
    return artist_name

VALID_ARTISTS = {normalize_artist_name(name) for name in RAW_VALID_ARTISTS}
TRANSLATION_ARTISTS = {normalize_artist_name(name) for name in RAW_TRANSLATION_ARTISTS}

def classify_artist(artist_name):
    normalized_artist = normalize_artist_name(artist_name)

    if normalized_artist in VALID_ARTISTS:
        return "valid"

    if normalized_artist in TRANSLATION_ARTISTS:
        return "translation"

    return "invalid"

In [9]:
def is_instrumental(track_name):
    return "instrumental" in str(track_name).lower()

def empty_result(status):
    return {
        "genius_title": None,
        "genius_artist": None,
        "genius_url": None,
        "lyrics": None,
        "title_score": 0,
        "artist_ok": False,
        "search_status": status
    }

def evaluate_match(track_name, song):
    title_score = similarity(track_name, song.title)
    artist_class = classify_artist(song.artist)
    artist_ok = artist_class == "valid"

    if artist_class == "valid" and title_score >= 0.75:
        status = "exact"
    elif artist_class == "translation":
        status = "manual_review"
    elif artist_class == "valid" and title_score >= 0.45:
        status = "manual_review"
    else:
        status = "rejected"

    return {
        "genius_title": song.title,
        "genius_artist": song.artist,
        "genius_url": song.url,
        "lyrics": song.lyrics,
        "title_score": round(title_score, 3),
        "artist_ok": artist_ok,
        "search_status": status
    }

def search_genius_track(track_name, artist_name="BTS"):
    if is_instrumental(track_name):
        return empty_result("instrumental")

    candidates = []
    candidate_ids = set()

    first_song = genius.search_song(track_name, artist_name)
    if first_song is not None:
        candidates.append(first_song)
        first_song_id = getattr(first_song, "id", None)
        if first_song_id is not None:
            candidate_ids.add(first_song_id)

    needs_fallback = True
    if candidates:
        first_result = evaluate_match(track_name, candidates[0])
        needs_fallback = first_result["search_status"] != "exact"

    if needs_fallback:
        search_response = genius.search_songs(f"{track_name} {artist_name}", per_page=5)
        for hit in search_response.get("hits", []):
            result = hit.get("result", {})
            song_id = result.get("id")
            if song_id is None or song_id in candidate_ids:
                continue

            candidate_song = genius.search_song(song_id=song_id)
            if candidate_song is None:
                continue

            candidates.append(candidate_song)
            candidate_ids.add(song_id)

    if not candidates:
        return empty_result("not_found")

    status_rank = {
        "exact": 3,
        "manual_review": 2,
        "rejected": 1,
        "not_found": 0,
        "instrumental": 0
    }

    scored_candidates = [evaluate_match(track_name, song) for song in candidates]
    best_match = max(
        scored_candidates,
        key=lambda item: (
            status_rank[item["search_status"]],
            item["title_score"],
            len(normalize_title(item["genius_title"]))
        )
    )

    return best_match

In [10]:
for title in ["ON", "Dynamite", "Black Swan", "Blood Sweat & Tears"]:
    result = search_genius_track(title)

    print("\n---")
    print(title)
    print(result["genius_title"])
    print(result["genius_artist"])
    print(result["title_score"])
    print(result["artist_ok"])
    print(result["search_status"])


---
ON
ON
BTS
1.0
True
exact

---
Dynamite
Dynamite
BTS
1.0
True
exact

---
Black Swan
Black Swan
BTS
1.0
True
exact

---
Blood Sweat & Tears
피 땀 눈물 (Blood Sweat & Tears) - Live
BTS
1.0
True
exact


In [11]:
tricky_titles = [
    "Blood Sweat & Tears",
    "Dimple",
    "The Truth Untold",
    "21st Century Girl",
    "MIC Drop (Steve Aoki Remix) (Full Length Edition)",
    "No More Dream - Japanese Ver.",
    "Hormone Sensou - Japanese Ver.",
    "Euphoria",
    "Moon",
    "Filter",
    "Intro: Boy Meets Evil"
]

sample_tracks = (
    tracks[tracks["track_name"].isin(tricky_titles)]
    .drop_duplicates(subset=["track_name"])
    .copy()
)

results = []

for _, row in sample_tracks.iterrows():
    result = search_genius_track(row["track_name"])

    results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        **result
    })

lyrics_sample = pd.DataFrame(results)

lyrics_sample[[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "artist_ok",
    "search_status"
]].sort_values("track_name").reset_index(drop=True)

,track_name,genius_title,genius_artist,title_score,artist_ok,search_status
0,21st Century Girl,21세기 소녀 (21st Century Girl),BTS,1.0,True,exact
1,Blood Sweat & Tears,피 땀 눈물 (Blood Sweat & Tears) - Live,BTS,1.0,True,exact
2,Dimple,보조개 (Dimple),BTS,1.0,True,exact
3,Euphoria,Euphoria,Jung Kook (정국),1.0,True,exact
4,Filter,Filter,Jimin (지민),1.0,True,exact
5,Hormone Sensou - Japanese Ver.,ホルモン戦争 (Hormone Sensou/War of Hormone) (Japane...,BTS,1.0,True,exact
6,Intro: Boy Meets Evil,Intro: Boy Meets Evil,j-hope,1.0,True,exact
7,MIC Drop (Steve Aoki Remix) (Full Length Edition),MIC Drop [Steve Aoki Remix] (Full Length Edition),BTS,1.0,True,exact
8,Moon,Moon,Jin (진),1.0,True,exact
9,No More Dream - Japanese Ver.,No More Dream (Japanese Ver.),BTS,1.0,True,exact


In [12]:
lyrics_sample["search_status"].value_counts()

search_status
exact    11
Name: count, dtype: int64

In [13]:
lyrics_sample[
    lyrics_sample["search_status"] != "exact"
][["track_name", "genius_title", "genius_artist", "title_score", "artist_ok", "search_status"]]

,track_name,genius_title,genius_artist,title_score,artist_ok,search_status


In [14]:
sample_tracks = tracks.sample(30, random_state=42).copy()

In [15]:
is_instrumental("SWIM (Instrumental)"), is_instrumental("Dynamite")

(True, False)

In [16]:
sample_tracks = tracks.sample(30, random_state=42).copy()

results = []

for _, row in sample_tracks.iterrows():
    result = search_genius_track(row["track_name"])

    results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        **result
    })

lyrics_sample = pd.DataFrame(results)

lyrics_sample[[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "artist_ok",
    "search_status"
]]

,track_name,genius_title,genius_artist,title_score,artist_ok,search_status
0,Dimple,보조개 (Dimple),BTS,1.00,True,exact
1,Shingeki no Boudan - Japanese Ver.,BTS - 進撃の防弾 (The Rise of Bangtan) (Japanese Ve...,Genius Romanizations,1.00,False,manual_review
2,MIC Drop (Steve Aoki Remix) (Full Length Edition),MIC Drop [Steve Aoki Remix] (Full Length Edition),BTS,1.00,True,exact
3,No More Dream - Japanese Ver.,No More Dream (Japanese Ver.),BTS,1.00,True,exact
4,Dynamite (Tropical Remix),Dynamite (Tropical Remix),BTS,1.00,True,exact
5,Stay Gold,Stay Gold,BTS,1.00,True,exact
6,Run,RUN,BTS,1.00,True,exact
7,Respect,Respect,BTS,1.00,True,exact
8,A Brand New Day (BTS World Original Soundtrack...,New Music Friday 06/14/19,Spotify,0.35,False,rejected
9,Make It Right (feat. Lauv) (Acoustic Remix),Make It Right (Acoustic Remix),BTS,1.00,True,exact


In [17]:
lyrics_sample["search_status"].value_counts()

search_status
exact            25
manual_review     3
rejected          1
instrumental      1
Name: count, dtype: int64

In [18]:
lyrics_sample[
    lyrics_sample["search_status"] != "exact"
][["track_name", "genius_title", "genius_artist", "title_score", "artist_ok", "search_status"]]

,track_name,genius_title,genius_artist,title_score,artist_ok,search_status
1,Shingeki no Boudan - Japanese Ver.,BTS - 進撃の防弾 (The Rise of Bangtan) (Japanese Ve...,Genius Romanizations,1.00,False,manual_review
8,A Brand New Day (BTS World Original Soundtrack...,New Music Friday 06/14/19,Spotify,0.35,False,rejected
10,Savage Love (Laxed – Siren Beat) [BTS Remix] (...,NaN,NaN,0.00,False,instrumental
26,My Universe - Galantis Remix,Coldplay & BTS - My Universe (Galantis Remix) ...,Genius Romanizations,1.00,False,manual_review
29,Iine!Pt.2 - Ano Bashode -,BTS - いいね!Pt.2 ～あの場所で～ (Like Pt.2 ~In That Pla...,Genius Romanizations,1.00,False,manual_review


In [19]:
sample_tracks_100 = tracks.sample(100, random_state=123).copy()

results = []

for _, row in sample_tracks_100.iterrows():
    result = search_genius_track(
        row["track_name"],
        "BTS"
    )

    results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        **result
    })

lyrics_sample_100 = pd.DataFrame(results)

In [20]:
lyrics_sample_100["search_status"].value_counts()

search_status
exact            88
rejected          4
instrumental      4
manual_review     2
not_found         2
Name: count, dtype: int64

In [21]:
lyrics_sample_100[
    lyrics_sample_100["search_status"] != "exact"
][[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "search_status"
]]

,track_name,genius_title,genius_artist,title_score,search_status
5,Bad Decisions (with BTS & Snoop Dogg),April 2021 Singles Release Calendar,Genius,0.297,rejected
23,Iine!,winx club,keiju (FIN),0.308,rejected
28,Old Town Road (feat. RM of BTS) - Seoul Town R...,Old Town Road (Seoul Town Road Remix),Lil Nas X,1.000,rejected
34,Iine!Pt.2 - Ano Bashode -,BTS - いいね!Pt.2 ～あの場所で～ (Like Pt.2 ~In That Pla...,Genius Romanizations,1.000,manual_review
49,A Brand New Day (BTS World Original Soundtrack...,New Music Friday 06/14/19,Spotify,0.350,rejected
50,Permission to Dance (Instrumental),NaN,NaN,0.000,instrumental
55,Permission to Dance (Instrumental),NaN,NaN,0.000,instrumental
56,My Universe - Galantis Remix,Coldplay & BTS - My Universe (Galantis Remix) ...,Genius Romanizations,1.000,manual_review
70,Skit: One Night in a Strange City,NaN,NaN,0.000,not_found
75,Butter (Instrumental),NaN,NaN,0.000,instrumental


In [22]:
status_counts = lyrics_sample_100["search_status"].value_counts(normalize=True) * 100
print(status_counts.round(1))

search_status
exact            88.0
rejected          4.0
instrumental      4.0
manual_review     2.0
not_found         2.0
Name: proportion, dtype: float64


In [23]:
from time import sleep
from tqdm.notebook import tqdm

RAW_DIR.mkdir(parents=True, exist_ok=True)

all_results = []

for _, row in tqdm(tracks.iterrows(), total=len(tracks)):
    result = search_genius_track(row["track_name"], "BTS")

    all_results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        "genius_title": result["genius_title"],
        "genius_artist": result["genius_artist"],
        "genius_url": result["genius_url"],
        "lyrics": result["lyrics"],
        "title_score": result["title_score"],
        "artist_ok": result["artist_ok"],
        "search_status": result["search_status"]
    })

    sleep(0.5)

  0%|          | 0/441 [00:00<?, ?it/s]

In [24]:
genius_lyrics_raw = pd.DataFrame(all_results)

genius_lyrics_raw.to_csv(
    RAW_DIR / "genius_lyrics_raw.csv",
    index=False
)

print("Saved:", RAW_DIR / "genius_lyrics_raw.csv")
print("Shape:", genius_lyrics_raw.shape)

Saved: ../data/raw/genius_lyrics_raw.csv
Shape: (441, 11)


In [25]:
genius_lyrics_raw["search_status"].value_counts()

search_status
exact            381
rejected          27
manual_review     12
not_found         11
instrumental      10
Name: count, dtype: int64

In [26]:
genius_lyrics_raw[
    genius_lyrics_raw["search_status"] != "exact"
][[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "artist_ok",
    "search_status"
]]

,track_name,genius_title,genius_artist,title_score,artist_ok,search_status
8,SWIM (Instrumental),NaN,NaN,0.000,False,instrumental
14,No. 29,BTS on No. 29,BTS,0.588,True,manual_review
83,Skit,NaN,NaN,0.000,False,not_found
100,OUTRO : The Journey,The Journey,Beth Brookfield,0.786,False,rejected
120,ON (Feat. Sia),ON [BTS/SIA COVER],UCLA Bruin Marching Band,1.000,False,rejected
158,Paradise,Paradise,Muze Sikk,1.000,False,rejected
182,Skit: Billboard Music Awards Speech,Top 100 Rap Songs of 2013,Rap Genius,0.136,False,rejected
222,Dope - Chou Yabee! - - Japanese Ver.,BTS - DOPE (超ヤベー) [Japanese Ver.] (Romanized),Genius Romanizations,1.000,False,manual_review
225,Funtan Syounendan - Japanese Ver.,NaN,NaN,0.000,False,not_found
261,Skit: One Night in a Strange City,NaN,NaN,0.000,False,not_found


In [27]:
check = pd.read_csv(RAW_DIR / "genius_lyrics_raw.csv")

print(check.shape)
check["search_status"].value_counts()

(441, 11)


search_status
exact            381
rejected          27
manual_review     12
not_found         11
instrumental      10
Name: count, dtype: int64